# Regime A calibration (tuning)

Turn the `WorkloadConfig` knobs until `generate()` output passes the frozen
`spec/regime_A.json` tolerances. `validate()` returns both pass/fail **and** the
MSM objective **Q** to minimize.

Workflow: edit the **Tweak** cell -> re-run it -> watch Q drop -> freeze when seeds pass.

In [ ]:
import dataclasses
from pathlib import Path
import numpy as np
import workload_gen as wg

# Robust to cwd: locate the frozen spec next to the package, not via a relative path.
SPEC = Path(wg.__file__).parent / "spec" / "regime_A.json"

def evaluate(cfg, seeds=range(8), show=True):
    """Generate + validate over several seeds; report per-check pass-rate with
    the current mean value next to the spec target (what it SHOULD be).
    Judge a setting on the pass-rate across seeds, NEVER one lucky draw (seed 0).
    Returns (mean Q, reports)."""
    reports = [wg.validate(wg.generate(cfg, seed=s), SPEC) for s in seeds]
    Qs = [r.distance for r in reports]
    n = len(reports)
    if show:
        print(f"per-check over {n} seeds   (current mean vs spec target):")
        for i, c in enumerate(reports[0].checks):
            rate = sum(r.checks[i].passed for r in reports)
            cur = np.mean([r.checks[i].value for r in reports])
            flag = "[pass]" if rate == n else "[fail]"
            print(f"  {flag} {c.name:<22} cur {cur:< 11.4g} target {c.target:< 11.4g}  {rate}/{n}")
        n_all = sum(r.passed for r in reports)
        print(f"mean Q = {np.mean(Qs):.4g} (min {min(Qs):.3g}, max {max(Qs):.3g}) "
              f"| all-checks-pass: {n_all}/{n}")
    return float(np.mean(Qs)), reports

## Knob -> stat reference

| If this stat is off | Turn this knob | Direction |
|---|---|---|
| kurtosis >> 25, ramp_abs_mean too low | `_job_profile` (generator.py) + small `noise_amp_W` | up: within-job dynamics |
| per_rack_mean fails (sampling noise, ~20% at 1 day) | `duration` sigma | down: lighter tail -> smoother per-rack means (also lowers kurtosis) |
| corr / pc1 too low | `job_size` (beta) | narrower / higher mean -> more synchronization |
| total_mean / per_rack level | `arrival_rate_per_s` (occupancy) | scale to 16.1 MW |
| marginal too idle (median low) | `arrival_rate_per_s` x `duration` | up: occupancy |

Notes:
- `_job_profile` (intra-job shape) is **not** a cfg knob -- it's a function in `generator.py`.
- Tradeoff: raw `noise_amp_W` raises ramp_abs_mean but **lowers corr** -> prefer a shaped `_job_profile` (correlated within-job variation) which adds ramps *without* killing corr.
- `per_rack_mean` mean-level bias was fixed in `regime_A_starting`; its remaining failure is sampling noise -> attack via `duration` sigma, and judge on the multi-seed pass-rate.

In [9]:
# Baseline: the constraint-derived starting theta
cfg = wg.WorkloadConfig.regime_A_starting(SPEC)
evaluate(cfg);

Validation: FAIL   Q(theta) = 18.64
  [FAIL] per_rack_mean          =  5.2377       (max |dev| 5.2% <= 5%)
  [PASS] total_mean_W           =  1.6768e+07   (16.77 MW vs 16.12 +/-7%)
  [PASS] pc1_var_share          =  0.98027      (>= 0.98)
  [PASS] ramp_excess_kurtosis   =  133.87       (>= 15)
  [PASS] offdiag_corr_mean      =  0.97782      (in [0.95, 0.999])
mean Q over 5 seeds = 23.21 (min 5.92, max 40.9) | passing: 0/5


### Tuning
- **Keep seeds fixed** while turning one knob, so Q changes only from your edit (common random numbers).
- A setting is "passing" only if it passes across **multiple seeds**.
- Read the **dominant squared term** in Q to choose the next knob.
- Knobs **interact** (occupancy moves both total_mean and busy-fraction) -- adjust one at a time.

In [10]:
# === TWEAK CELL: edit knobs, re-run, watch Q ===
cfg = wg.WorkloadConfig.regime_A_starting(SPEC)        # start fresh (comment out to keep tuning the same cfg)

cfg = dataclasses.replace(                             # replace() rebuilds -> re-runs validation
    cfg,
    noise_amp_W = 30_000.0,
    # arrival_rate_per_s = 3.0e-4,
    # job_power = wg.DistSpec("normal", {"mean": 756_000, "std": 50_000}),
    # duration  = wg.DistSpec("lognormal", {"mu": 7.62, "sigma": 1.0}),
    # job_size  = wg.DistSpec("beta", {"a": 30, "b": 1.5}),
    # placement = "contiguous",
)
evaluate(cfg);

Validation: FAIL   Q(theta) = 2.694
  [FAIL] per_rack_mean          =  5.3641       (max |dev| 5.4% <= 5%)
  [PASS] total_mean_W           =  1.6767e+07   (16.77 MW vs 16.12 +/-7%)
  [FAIL] pc1_var_share          =  0.97718      (>= 0.98)
  [PASS] ramp_excess_kurtosis   =  66.478       (>= 15)
  [PASS] offdiag_corr_mean      =  0.97448      (in [0.95, 0.999])
mean Q over 5 seeds = 3.231 (min 1.76, max 5.04) | passing: 0/5


In [ ]:
# Freeze the calibrated config once seeds pass
out = Path(wg.__file__).parent / "spec" / "regime_A_calib.json"
cfg.to_json(out)
print("saved", out)